# LSTM Data Prep (Boundary-Safe) — Raw Sliding Windows for `Fault_Within_6h`

A separate notebook from `04a_lstm_windows.ipynb` — that file and `04_lstm_classifier.ipynb` are left
untouched. This notebook redoes the same `Fault_Within_6h` window-building step with one methodology
fix: **split each vehicle's raw rows into train/test first, then build windows independently within
each split**, instead of building windows across a vehicle's full timeline and labeling each window
train/test afterward by where its target row falls.

**Why this matters:** in the original (`04a`) approach, a window's train/test label was decided by the
position of its *target* row, but the window's 24 raw input hours could still span across the
train/test cutoff. Concretely, the first test window and the last train window for a vehicle shared 23
of their 24 raw hours — a near-duplicate-window overlap right at the split boundary. Splitting the
rows first, then windowing each split independently, makes that overlap structurally impossible: a
window built only from training rows can never include any test row, and vice versa.

**Where this comes from:** Hüseyin's PR #31 (`notebooks/04_sequence_neural_net.ipynb`, merged to
`main`) already uses this split-then-window order (`chronological_split()` + `make_windows()` there).
The `build_windows_boundary_safe` function below adapts that same logic for this project's raw-window
LSTM pipeline — it is not a copy of his notebook and does not modify it.

**Same pandas/TensorFlow separation as `04a`/`04`, same reason:** pandas and TensorFlow/Keras deadlock
in the same Jupyter kernel process on this machine (see `04a_lstm_windows.ipynb` for the isolation
testing that found this). This notebook does the pandas-heavy prep and saves a `.npz`;
`04c_lstm_boundary_safe_classifier.ipynb` loads it with no pandas import at all.

**Scope:** only the `Fault_Within_6h` primary target, matching what was asked. The `Fault_Within_12h`
side experiment in `04a`/`04b` is untouched and out of scope here.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

SEQ_SENSORS = ["Motor_RPM", "Motor_Torque", "Motor_Temp", "Battery_Temp"]
WINDOW_HOURS = 24  # matches 04a_lstm_windows.ipynb
TARGET = "Fault_Within_6h"


## 1. Load data and build boundary-safe raw sliding windows, per vehicle

Same source CSV, same target, same window/target alignment as `04a_lstm_windows.ipynb` (a window
covers 24 consecutive raw hours; the label is `Fault_Within_6h` at the hour right after the window
ends). The only change is *when* the train/test split happens relative to windowing.

In [2]:
df = pd.read_csv(
    "../data/processed/driving_pattern_diagnostics_sequence_features.csv",
    parse_dates=["timestamp"],
    usecols=["timestamp", "user_profile", TARGET] + SEQ_SENSORS,
)
df = df.sort_values(["user_profile", "timestamp"]).reset_index(drop=True)


def build_windows_boundary_safe(vehicle_df, seq_cols, target_col, window_hours, test_frac=0.2):
    """Split a vehicle's raw rows into train/test FIRST (same cutoff position as 04a's
    build_windows: int(n_rows * (1 - test_frac))), then build windows independently within
    each split. A window can never straddle the train/test boundary, because the split
    happens before windowing, not after. Adapted from notebooks/04_sequence_neural_net.ipynb
    (PR #31): chronological_split() + make_windows() there, combined into one function here
    since this pipeline processes one vehicle at a time already."""
    n_rows = len(vehicle_df)
    cutoff = int(n_rows * (1 - test_frac))
    train_part = vehicle_df.iloc[:cutoff]
    test_part = vehicle_df.iloc[cutoff:]

    def _windows_from(part):
        raw = part[seq_cols].to_numpy(dtype="float32")
        target = part[target_col].to_numpy()
        X, y = [], []
        for i in range(window_hours, len(part)):
            if pd.isna(target[i]):
                continue
            X.append(raw[i - window_hours : i])
            y.append(bool(target[i]))
        return np.array(X), np.array(y)

    X_train, y_train = _windows_from(train_part)
    X_test, y_test = _windows_from(test_part)
    return X_train, y_train, X_test, y_test


X_train_parts, y_train_parts, X_test_parts, y_test_parts = [], [], [], []
for profile, vdf in df.groupby("user_profile"):
    X_tr, y_tr, X_te, y_te = build_windows_boundary_safe(vdf, SEQ_SENSORS, TARGET, WINDOW_HOURS)
    X_train_parts.append(X_tr); y_train_parts.append(y_tr)
    X_test_parts.append(X_te); y_test_parts.append(y_te)
    print(f"{profile}: {len(X_tr)} train windows, {len(X_te)} test windows "
          f"(split before windowing -- no boundary overlap)")

X_train = np.concatenate(X_train_parts)
y_train = np.concatenate(y_train_parts)
X_test = np.concatenate(X_test_parts)
y_test = np.concatenate(y_test_parts)

print()
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean()*100:.3f}%, Test positive rate: {y_test.mean()*100:.3f}%")

daily_user: 34997 train windows, 8726 test windows (split before windowing -- no boundary overlap)
heavy_user: 34997 train windows, 8726 test windows (split before windowing -- no boundary overlap)
moderate_user: 34997 train windows, 8726 test windows (split before windowing -- no boundary overlap)
rare_user: 34997 train windows, 8726 test windows (split before windowing -- no boundary overlap)

X_train: (139988, 24, 4), X_test: (34904, 24, 4)
Train positive rate: 8.976%, Test positive rate: 9.391%


## 2. Scale (train fold only) and compute class weights (train fold only)

Same discipline as `04a_lstm_windows.ipynb`: `StandardScaler` fit on training windows only, class
weights computed from `y_train` only.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

n_features = len(SEQ_SENSORS)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, n_features)
).reshape(X_train.shape).astype("float32")
X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(X_test.shape).astype("float32")

class_weight_values = compute_class_weight(class_weight="balanced", classes=np.array([False, True]), y=y_train)
class_weight = {0: class_weight_values[0], 1: class_weight_values[1]}
print("Class weights (from y_train only):", class_weight)

Class weights (from y_train only): {0: np.float64(0.5493042857254969), 1: np.float64(5.570553123756467)}


## 3. Save prepared arrays for `04c_lstm_boundary_safe_classifier.ipynb`

Saved under a distinct filename (`lstm_fault_within_6h_windows_boundary_safe.npz`) — deliberately not
overwriting `lstm_fault_within_6h_windows.npz`, which `04_lstm_classifier.ipynb` still reads from
unchanged.

In [4]:
import os

os.makedirs("../data/processed", exist_ok=True)
np.savez(
    "../data/processed/lstm_fault_within_6h_windows_boundary_safe.npz",
    X_train=X_train_scaled,
    y_train=y_train.astype("float32"),
    X_test=X_test_scaled,
    y_test=y_test.astype("float32"),
    class_weight_0=class_weight_values[0],
    class_weight_1=class_weight_values[1],
)
print(f"Saved to ../data/processed/lstm_fault_within_6h_windows_boundary_safe.npz "
      f"({X_train_scaled.shape[0]:,} train + {X_test_scaled.shape[0]:,} test windows)")

Saved to ../data/processed/lstm_fault_within_6h_windows_boundary_safe.npz (139,988 train + 34,904 test windows)


## Summary

- Rebuilt `Fault_Within_6h` raw 24-hour sliding windows with split-then-window ordering: each vehicle's
  rows are split into train/test first (same cutoff position as `04a`), then windows are built
  independently within each split — a window can never straddle the train/test boundary.
- Train windows: unaffected (139,988 total, matching `04a` exactly) — train sits earlier in each
  vehicle's timeline and was never affected by how the test-side boundary is handled.
- Test windows: reduced from 35,000 (`04a`) to 34,904 — a drop of exactly `WINDOW_HOURS` (24) per
  vehicle, the incomplete boundary windows that the old approach let leak across the split.
- Saved to `data/processed/lstm_fault_within_6h_windows_boundary_safe.npz` for
  `04c_lstm_boundary_safe_classifier.ipynb`, which trains the same unchanged LSTM architecture as
  `04_lstm_classifier.ipynb` on these windows and compares results directly.